# Step 0 — Data Discovery (YAMBDA-50m)

Снимаем риски до написания кода в `src/`. Цель — получить реальные числа, на которые опираются последующие шаги Phase 1: `n_users`, `n_items`, медиана длины истории (для `max_seq_len`), валидация GTS-границ, оценка `d_a` audio-эмбеддинга.

**Что фиксируем в `docs/phase_1_log.md` после прогона:**
- Total events, event_type breakdown, played_ratio_pct distribution
- После фильтра `listen & played_ratio_pct>=50`: `n_users`, `n_items`, `n_events`
- Per-user history length: median, 95p, 99p → решает `max_seq_len`
- Timestamp range + GTS sanity (val/test users counts)
- Audio embed dim (через HfFileSystem range-read) или TBD
- Estimated audio subset size

**Окружение:** локально на M4 Pro (24 GB). Если упадёт по памяти на groupby — fallback `sort_values + np.diff`.

## 1. Imports & setup

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# YAMBDA Constants (из references/yambda/benchmarks/yambda/constants.py)
TEST_TIMESTAMP = 26000000 - 86400  # последний день — test
VAL_SIZE = 86400                    # 1 day
GAP_SIZE = 1800                     # 30 min
TRACK_LISTEN_THRESHOLD = 50         # played_ratio_pct >= 50% = считается прослушиванием

## 2. Load YAMBDA-50m flat-multievent

Используем `flat-multievent-50m` (как в `playground.ipynb`) — нужен flat для time-based GTS.

Ожидание: ~47.8M строк, ~1.6 GB в pandas (uint32/uint8 dtypes).

In [ ]:
ds = load_dataset("yandex/yambda", "flat-multievent-50m", split="train")
df = ds.to_pandas()
del ds
gc.collect()

print(f"Total events: {len(df):,}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.info()

In [ ]:
df.head(5)

## 3. Event-type breakdown

Хотим увидеть пропорции listen / like / unlike / dislike / undislike / multi_event. Нужно для понимания, сколько данных переживёт фильтр и что доступно для будущих сигналов (likes как ground truth для групп).

In [ ]:
event_counts = df['event_type'].value_counts()
event_share = (event_counts / len(df) * 100).round(2)
print("event_type counts:")
print(pd.concat([event_counts, event_share.rename('share_pct')], axis=1))

## 4. Распределение `played_ratio_pct` (для listen)

Валидируем порог 50%. Хотим увидеть, какая доля listen-событий имеет `played_ratio_pct >= 50%` (= валидное прослушивание для скорера).

In [ ]:
listen_mask = df['event_type'] == 'listen'
ratio_listen = df.loc[listen_mask, 'played_ratio_pct']

print("played_ratio_pct (listen events) describe:")
print(ratio_listen.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))
print()
for thr in [25, 50, 75, 90]:
    share = (ratio_listen >= thr).mean() * 100
    print(f"  share with played_ratio_pct >= {thr:>2}%: {share:5.2f}%")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(ratio_listen, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(50, color='red', linestyle='--', label='threshold=50%')
ax.set_xlabel('played_ratio_pct')
ax.set_ylabel('count')
ax.set_title('played_ratio_pct distribution (listen events)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Применяем фильтр: listen & played_ratio_pct >= 50

Это будет основной набор для обучения скорера. Считаем `n_users`, `n_items`, `n_events`.

In [ ]:
filt = (df['event_type'] == 'listen') & (df['played_ratio_pct'] >= TRACK_LISTEN_THRESHOLD)
listens = df.loc[filt, ['uid', 'item_id', 'timestamp']].reset_index(drop=True)

n_events = len(listens)
n_users = listens['uid'].nunique()
n_items = listens['item_id'].nunique()

print(f"After filter (listen & played_ratio_pct>={TRACK_LISTEN_THRESHOLD}):")
print(f"  n_events: {n_events:,}")
print(f"  n_users:  {n_users:,}")
print(f"  n_items:  {n_items:,}")
print(f"  events/user (mean): {n_events / n_users:.1f}")
print(f"  events/item (mean): {n_events / n_items:.1f}")

## 6. Per-user history length

Решает выбор `max_seq_len` для gSASRec. План — 200, но если 99p < 100, то 200 избыточно.

In [ ]:
user_hist = listens.groupby('uid', sort=False).size()

print("per-user history length (post-filter):")
print(user_hist.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(1))

for q in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]:
    print(f"  q={q:5.3f}: {user_hist.quantile(q):.0f}")

print()
for thr in [10, 50, 100, 200, 500, 1000]:
    share = (user_hist >= thr).mean() * 100
    print(f"  users with >= {thr:>4} events: {share:5.2f}%")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(user_hist.values, bins=100, edgecolor='black', alpha=0.7, range=(0, 500))
ax.set_xlabel('history length (capped at 500 for viz)')
ax.set_ylabel('users count')
ax.set_yscale('log')
ax.set_title('Per-user history length distribution (post-filter)')
plt.tight_layout()
plt.show()

## 7. Timestamp distribution + GTS sanity

Валидируем GTS-границы из yambda Constants (TEST_TIMESTAMP = 26000000-86400, val_size=1 day, gap=30min). Хотим убедиться, что в val/test попадает достаточно user'ов (>1000).

In [ ]:
ts = listens['timestamp']
print(f"timestamp range: [{ts.min()}, {ts.max()}]")
print(f"  TEST_TIMESTAMP = {TEST_TIMESTAMP}")
print(f"  diff to ts.max(): {ts.max() - TEST_TIMESTAMP}")
print()

# GTS boundaries
test_start = TEST_TIMESTAMP
val_end = test_start - GAP_SIZE
val_start = val_end - VAL_SIZE
train_end = val_start - GAP_SIZE

print("GTS boundaries:")
print(f"  train: ts < {train_end}")
print(f"  val:   {val_start} <= ts <= {val_end}")
print(f"  test:  ts >= {test_start}")
print()

train_mask = listens['timestamp'] < train_end
val_mask = (listens['timestamp'] >= val_start) & (listens['timestamp'] <= val_end)
test_mask = listens['timestamp'] >= test_start

n_train = train_mask.sum()
n_val = val_mask.sum()
n_test = test_mask.sum()
n_gap = n_events - n_train - n_val - n_test

u_train = listens.loc[train_mask, 'uid'].nunique()
u_val = listens.loc[val_mask, 'uid'].nunique()
u_test = listens.loc[test_mask, 'uid'].nunique()

print(f"events  — train: {n_train:>12,}, val: {n_val:>10,}, test: {n_test:>10,}, gap: {n_gap:>10,}")
print(f"users   — train: {u_train:>12,}, val: {u_val:>10,}, test: {u_test:>10,}")

# Sanity: val users должны быть подмножеством train
val_users = set(listens.loc[val_mask, 'uid'])
train_users = set(listens.loc[train_mask, 'uid'])
print(f"\nval users ⊆ train users: {val_users.issubset(train_users)}")
print(f"val \\ train: {len(val_users - train_users):,} users (cold-start в val)")

Если `u_val < 1000` — увеличить `VAL_SIZE` до 2 дней (172800).

## 8. Audio embed dim через HfFileSystem (range-read)

Цель: узнать `d_a` без скачивания 14GB. Используем `HfFileSystem` + `pyarrow.parquet.ParquetFile` — он умеет HTTP range requests через fsspec.

Если упадёт (timeouts, ошибки range) — оставляем `d_a = TBD` до Phase 2 (на Colab будет полный subset extraction).

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

audio_dim = None
audio_n_rows = None

try:
    fs = HfFileSystem()
    # Точный путь embeddings.parquet — проверим список файлов в репо
    files = fs.ls("datasets/yandex/yambda", detail=False)
    print("Top-level files:")
    for f in files:
        print(" ", f)
except Exception as e:
    print(f"HfFileSystem ls failed: {e}")

In [ ]:
# Если embeddings.parquet виден — пробуем прочитать schema без скачивания
try:
    fs = HfFileSystem()
    embed_path = "datasets/yandex/yambda/embeddings.parquet"
    print(f"Trying: {embed_path}")

    with fs.open(embed_path, "rb") as f:
        pf = pq.ParquetFile(f)
        print("Schema:")
        print(pf.schema_arrow)
        print()
        print(f"num_row_groups: {pf.num_row_groups}")
        print(f"total rows (metadata): {pf.metadata.num_rows:,}")
        audio_n_rows = pf.metadata.num_rows

        # Читаем только первый row group (минимальный range request)
        rg = pf.read_row_group(0)
        sample = rg.slice(0, 3).to_pandas()
        print("\nFirst 3 rows:")
        print(sample)

        # Извлекаем dim из первого вектора
        for col in sample.columns:
            val = sample[col].iloc[0]
            if hasattr(val, '__len__') and not isinstance(val, str):
                audio_dim = len(val)
                print(f"\n=> Audio embed dim (column '{col}'): d_a = {audio_dim}")
                break
except Exception as e:
    import traceback
    print(f"Range-read failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\n=> Оставляем d_a = TBD до Phase 2 (Colab).")

## 9. Estimated audio subset size

Прикидываем, сколько весит subset аудиоэмбеддингов для items, реально появляющихся в `listens` (после фильтра).

In [ ]:
if audio_dim is not None:
    # float32 = 4 bytes/dim
    bytes_per_item = audio_dim * 4
    full_mb = audio_n_rows * bytes_per_item / 1e6 if audio_n_rows else None
    subset_mb = n_items * bytes_per_item / 1e6
    print(f"Per-item: {bytes_per_item} bytes ({audio_dim} dims × float32)")
    if full_mb:
        print(f"Full file (всего {audio_n_rows:,} items): ~{full_mb:.0f} MB ({full_mb/1024:.2f} GB)")
    print(f"Subset для наших {n_items:,} items: ~{subset_mb:.0f} MB ({subset_mb/1024:.2f} GB)")
else:
    print("d_a unknown — пропускаем оценку subset.")

## 10. Summary — числа для phase_1_log.md

Скопировать вывод этой ячейки в раздел «Data discovery findings».

In [ ]:
print("=" * 60)
print("DATA DISCOVERY SUMMARY (для копирования в phase_1_log.md)")
print("=" * 60)
print(f"Total events (50m, post-load): {len(df):,}")
print(f"Event_type breakdown:")
for et, cnt in event_counts.items():
    print(f"  {et:>15}: {cnt:>12,} ({cnt/len(df)*100:.2f}%)")
print()
print(f"played_ratio_pct (listen): median={ratio_listen.median():.1f}, 90p={ratio_listen.quantile(0.9):.1f}")
print()
print(f"After filter (listen & played_ratio_pct>=50):")
print(f"  n_events: {n_events:,}")
print(f"  n_users:  {n_users:,}")
print(f"  n_items:  {n_items:,}")
print()
print(f"Per-user history length (post-filter):")
print(f"  median: {user_hist.median():.0f}")
print(f"  95p:    {user_hist.quantile(0.95):.0f}")
print(f"  99p:    {user_hist.quantile(0.99):.0f}")
print(f"  max:    {user_hist.max():.0f}")
print()
print(f"Timestamp range: [{ts.min()}, {ts.max()}]")
print(f"GTS sanity (val_size={VAL_SIZE}, gap={GAP_SIZE}):")
print(f"  users in train: {u_train:,}")
print(f"  users in val:   {u_val:,}  {'OK' if u_val >= 1000 else 'TOO FEW — увеличить VAL_SIZE!'}")
print(f"  users in test:  {u_test:,}")
print()
print(f"Audio embed dim (d_a): {audio_dim if audio_dim else 'TBD (отложено до Phase 2)'}")
if audio_dim is not None:
    subset_mb = n_items * audio_dim * 4 / 1e6
    print(f"Estimated audio subset for {n_items:,} items: ~{subset_mb:.0f} MB")